# From ML I to ML II: A Recap Lab

**Machine Learning in Finance II — first class.** This lab revises the parts of *Machine Learning in Finance I* that the ML II lectures and labs build on. ML II begins with **boosting models** (AdaBoost, Gradient Boosting, XGBoost, LightGBM, CatBoost) and then moves to **neural networks** (MLP, RNN, attention, CNN). Those methods are **not** covered here. Instead, we refresh the material they depend on, so that the first lectures can focus on new ideas.

| What ML II relies on | Where you met it in ML I | Section |
|---|---|---|
| Loss vs. cost vs. evaluation metric; the derivative of a loss | Lecture 1 (cost functions), Lecture 2 (metrics) | 1 |
| Gradient descent: learning rate, feature scaling, batch / mini-batch / stochastic variants, the chain rule | Lecture 1, `linear_and_logistic_regression.ipynb` | 2 |
| Regularization as a penalty in the cost function | Lecture 4, `core_ml_techniques.ipynb` | 3 |
| Bias–variance trade-off, validation and learning curves | Lecture 2, `core_ml_techniques.ipynb` | 4 |
| Decision trees as building blocks: split search, leaf values, `sample_weight` | Lecture 5, `decision_trees_and_random_forest.ipynb` | 5 |
| Leakage-free pipelines, honest validation, probability metrics, calibration | Labs 1–5, Kaggle League | 6 |
| Validation for time-ordered data | Lecture 2, `core_ml_techniques.ipynb` | 7 |

**How to use this notebook**

- 🔁 marks a short recap of theory, ✍️ marks an exercise (an empty cell for your code, followed by a hidden solution), and 🔭 marks a note on *why* the concept matters in ML II.
- Sections 1–6 are the core of the class (about 90 minutes). Section 7 and the warm-up assignment are for self-study.
- The data split in Section 2 is fixed with a seed. The benchmark table at the end of Section 6 is the **reference that the ML II models will have to beat** on the same data.

Author: Michał Woźniak

## Technical Setup

This notebook is part of the course repository, which already ships a complete, locked environment (`pyproject.toml` + `uv.lock`). You do **not** need to install packages one by one.

```bash
# from the root of the repository
uv sync
uv run jupyter lab
```

A detailed, step-by-step installation guide is available in [`SETUP.md`](../SETUP.md).

**Packages used in this notebook:** `numpy`, `pandas`, `matplotlib`, `seaborn`, `scipy`, `scikit-learn`.

**Data:** *Default of Credit Card Clients* (`data/credit_card_default.csv.gz`, documented in [`data/README.md`](../data/README.md)) ships with the repository, so no internet connection is needed. The remaining examples use simulated data.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import loguniform, randint, uniform
from sklearn.base import clone
from sklearn.calibration import CalibrationDisplay
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import BaggingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    mean_squared_error,
    roc_auc_score,
)
from sklearn.model_selection import (
    KFold,
    RandomizedSearchCV,
    StratifiedKFold,
    TimeSeriesSplit,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.utils.class_weight import compute_sample_weight

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.precision", 4)

# 1. Loss, Cost and Evaluation Metric

🔁 **Recap (Lecture 1).** We assume $Y = f(X) + \varepsilon$ and estimate $\hat f$ by minimising how "wrong" the model is on the training data. Three objects are easy to confuse:

| Object | Definition | Role | Must be differentiable? |
|---|---|---|---|
| **Loss** $L(y, \hat y)$ | error of a *single* prediction | building block of the cost | yes, if we optimise with gradients |
| **Cost** $J(\theta)$ | $\frac{1}{n}\sum_{i=1}^n L(y_i, \hat y_i) + \lambda R(\theta)$ | the function the algorithm **minimises** | yes, if we optimise with gradients |
| **Evaluation metric** | ROC AUC, RMSE, F1, Brier, ... | used to **compare and select** models after (or during) training | no |

Four losses from ML I, written with the derivative with respect to the model output (we will need these derivatives in Section 2):

| Loss | Formula | Used by | Derivative w.r.t. the model output |
|---|---|---|---|
| Squared error | $\frac{1}{2}(y - \hat y)^2$ | OLS, Ridge, Lasso, regression trees | $\frac{\partial L}{\partial \hat y} = -(y - \hat y)$ |
| Absolute error | $\lvert y - \hat y \rvert$ | median regression, `criterion="absolute_error"` | $\frac{\partial L}{\partial \hat y} = -\operatorname{sign}(y - \hat y)$ |
| Log-loss (binary cross-entropy), $p = \sigma(z)$ | $-[y \log p + (1 - y)\log(1 - p)]$ | logistic regression | $\frac{\partial L}{\partial z} = p - y$ |
| Hinge, $y \in \{-1, +1\}$ | $\max(0,\, 1 - y\,z)$ | linear SVM | $\frac{\partial L}{\partial z} = -y\,\mathbb{1}(y z < 1)$ |

Here $z$ is the raw score of a classifier (the *logit* $z = x^\top w + b$ for logistic regression) and $\sigma(z) = 1 / (1 + e^{-z})$ is the sigmoid.

In [ ]:
def squared_loss(y, y_hat):
    return 0.5 * (y - y_hat) ** 2


def absolute_loss(y, y_hat):
    return np.abs(y - y_hat)


def binary_log_loss(y, p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)  # log(0) = -inf, so we never let p touch 0 or 1
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))


# sanity check against scikit-learn (which reports the MEAN loss, without the 1/2 for MSE)
y_true = np.array([0, 1, 1, 0, 1])
p_hat = np.array([0.1, 0.8, 0.6, 0.3, 0.95])
print(f"our mean log-loss: {binary_log_loss(y_true, p_hat).mean():.6f}")
print(f"sklearn log_loss : {log_loss(y_true, p_hat):.6f}")

y_reg = np.array([1.0, 2.0, 3.0])
y_reg_hat = np.array([1.5, 1.0, 3.5])
print(f"our 2 x mean squared loss: {2 * squared_loss(y_reg, y_reg_hat).mean():.6f}")
print(f"sklearn MSE              : {mean_squared_error(y_reg, y_reg_hat):.6f}")

The plots below show the shape of the losses. In the classification panel the horizontal axis is the **margin** $m = y\,z$ with $y \in \{-1, +1\}$: a positive margin means a correct classification, and a large positive margin means a confident, correct one. In this notation the log-loss is $\log(1 + e^{-m})$ (divided by $\log 2$ so that it passes through 1 at $m = 0$, like the hinge loss).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

residual = np.linspace(-3, 3, 400)
axes[0].plot(residual, 0.5 * residual**2, label=r"squared: $\frac{1}{2}r^2$")
axes[0].plot(residual, np.abs(residual), label=r"absolute: $|r|$")
axes[0].set(xlabel=r"residual $r = y - \hat{y}$", ylabel="loss", title="Regression losses", ylim=(0, 4))
axes[0].legend()

margin = np.linspace(-3, 3, 400)
axes[1].plot(margin, (margin < 0).astype(float), label="0–1 loss (what we count)", color="black", linestyle=":")
axes[1].plot(margin, np.maximum(0, 1 - margin), label="hinge (SVM)")
axes[1].plot(margin, np.log1p(np.exp(-margin)) / np.log(2), label="log-loss (logistic regression)")
axes[1].set(xlabel=r"margin $m = y \cdot z$", ylabel="loss", title="Classification losses", ylim=(0, 4))
axes[1].legend()

plt.tight_layout()
plt.show()

What to read from the plots:

- The **0–1 loss** is what we ultimately care about in classification, but it is flat almost everywhere and jumps at zero. Its gradient is zero or undefined, so we cannot optimise it with gradients. Hinge and log-loss are its smooth (or at least continuous) **surrogates**.
- The **hinge loss** is exactly zero once the margin exceeds 1: well-classified points stop influencing the SVM, and only the support vectors matter. The **log-loss** never reaches zero, so every observation keeps pulling on the solution.
- The **squared loss** grows quadratically, so a few large residuals (outliers, fat-tailed returns) dominate the cost. The **absolute loss** grows linearly and is more robust.

The log-loss also punishes **confident mistakes** very hard, which matters whenever probabilities are used downstream (credit risk, pricing):

In [ ]:
p_default = np.array([0.6, 0.9, 0.99, 0.999])
pd.DataFrame(
    {
        "predicted P(default)": p_default,
        "log-loss if the client did NOT default (y = 0)": binary_log_loss(0, p_default),
        "log-loss if the client defaulted (y = 1)": binary_log_loss(1, p_default),
    }
)

🔭 **Why this matters in ML II.** Every boosting library and every neural network framework asks you for two separate things: an **objective / loss** (what is minimised, must be differentiable) and an **evaluation metric** (what you monitor and select on). In the Kaggle League you were scored on ROC AUC and RMSE. Neither of them is what logistic regression or a tree actually minimises, and you will have to make this choice consciously for every ML II model.

**Exercise 1.** ROC AUC depends only on the **ranking** of the predictions, while log-loss depends on their **values**. Verify this:

1. Take `y_true` and `p_hat` defined above.
2. Create `p_sharp = p_hat ** 3`. This is a monotone transformation, so the ranking does not change.
3. Compute the ROC AUC and the log-loss of `p_hat` and of `p_sharp`.
4. Explain in one sentence why ROC AUC cannot be used as the training loss of a gradient-based model.

In [ ]:
# your solution here

<details>
<summary>Solution</summary>

```python
p_sharp = p_hat**3
for name, p in [("p_hat", p_hat), ("p_sharp", p_sharp)]:
    print(f"{name:8s} ROC AUC = {roc_auc_score(y_true, p):.3f}   log-loss = {log_loss(y_true, p):.3f}")
# ROC AUC is identical, log-loss differs: AUC only looks at the order of the scores.
# AUC is a step function of the scores (it changes only when two scores swap places),
# so its gradient is zero almost everywhere - gradient descent gets no signal from it.
```

</details>

# 2. Gradient Descent

🔁 **Recap (Lecture 1).** For most models the first-order condition $\nabla J(\theta) = 0$ has no closed-form solution, so we minimise the cost iteratively:

$$\theta^{(t+1)} = \theta^{(t)} - \eta\, \nabla_\theta J\big(\theta^{(t)}\big),$$

where $\eta$ is the **learning rate**. The three variants differ in how much data is used to compute the gradient in one update:

| Variant | Gradient computed on | One update is | Behaviour |
|---|---|---|---|
| **Batch** (vanilla) GD | all $n$ observations | exact, slow | smooth path, one update per pass over the data |
| **Mini-batch** GD | $B$ observations (e.g. 32–512) | noisy, fast | many updates per pass; the standard in practice |
| **Stochastic** GD (SGD) | 1 observation | very noisy, fastest | jumps around the minimum |

One full pass over the training data is called an **epoch**. Before each epoch the data is shuffled, so that the mini-batches differ between epochs.

## 2.1 Data for This Lab

We use the *Default of Credit Card Clients* dataset (30,000 clients of a Taiwanese bank, target `default_next_month`). It is split **once** into training (60%), validation (20%) and test (20%) sets, stratified by the target and with a fixed seed. The same split is used in Sections 2–6.

In [ ]:
df = pd.read_csv("../data/credit_card_default.csv.gz")
y = df.pop("default_next_month").to_numpy()
X = df

X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.25, stratify=y_tmp, random_state=RANDOM_STATE
)

print(f"train: {X_train.shape}, validation: {X_val.shape}, test: {X_test.shape}")
print(f"default rate - train: {y_train.mean():.3f}, validation: {y_val.mean():.3f}, test: {y_test.mean():.3f}")

For the from-scratch experiments we keep five numerical features on **very different scales**: the credit limit (in NT dollars), age (in years), the repayment status in the last month, the last bill and the last payment.

In [ ]:
GD_FEATURES = ["LIMIT_BAL", "AGE", "PAY_0", "BILL_AMT1", "PAY_AMT1"]

X_gd_raw = X_train[GD_FEATURES].to_numpy(dtype=float)
X_gd_val_raw = X_val[GD_FEATURES].to_numpy(dtype=float)

scaler_gd = StandardScaler().fit(X_gd_raw)  # fitted on the training set only
X_gd = scaler_gd.transform(X_gd_raw)
X_gd_val = scaler_gd.transform(X_gd_val_raw)

X_train[GD_FEATURES].describe().loc[["mean", "std", "min", "max"]]

## 2.2 The Gradient of the Log-Loss via the Chain Rule

🔁 **Recap.** For logistic regression the prediction is built in three steps: the linear score $z = x^\top w + b$, the probability $p = \sigma(z)$, and the loss $L(y, p)$. The **chain rule** gives the gradient with respect to the weights:

$$
\frac{\partial L}{\partial w}
= \underbrace{\frac{\partial L}{\partial p}}_{\frac{p - y}{p(1 - p)}}
\cdot \underbrace{\frac{\partial p}{\partial z}}_{p(1 - p)}
\cdot \underbrace{\frac{\partial z}{\partial w}}_{x}
= (p - y)\, x .
$$

The awkward factors $p(1-p)$ cancel, which is one reason why the sigmoid and the log-loss are used together. Averaged over $n$ observations and written in matrix form:

$$
\nabla_w J = \frac{1}{n} X^\top (p - y), \qquad \frac{\partial J}{\partial b} = \frac{1}{n}\sum_{i=1}^n (p_i - y_i).
$$

Pay attention to the **shapes**: $X$ is $(n \times d)$, $p - y$ is $(n,)$, so $X^\top (p - y)$ is $(d,)$ — one number per weight.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))  # clipping avoids overflow in exp


def cost_and_gradient(w, b, X, y, l2=0.0):
    # Mean log-loss (+ optional L2 penalty) and its gradient for logistic regression.
    p = sigmoid(X @ w + b)  # shape (n,)
    cost = binary_log_loss(y, p).mean() + 0.5 * l2 * (w @ w)
    error = p - y  # shape (n,)
    grad_w = X.T @ error / len(y) + l2 * w  # shape (d,)
    grad_b = error.mean()  # scalar (the intercept is not penalised)
    return cost, grad_w, grad_b

## 2.3 Gradient Check

Whenever you derive a gradient by hand, **check it numerically** before trusting it. The central finite difference approximates each partial derivative:

$$\frac{\partial J}{\partial w_k} \approx \frac{J(w + \epsilon e_k) - J(w - \epsilon e_k)}{2\epsilon},$$

where $e_k$ is the $k$-th unit vector. If the analytical gradient is correct, the relative difference between the two is tiny (around $10^{-7}$ or smaller). A mistake in the derivation typically shows up as a difference of order $10^{-2}$ or larger.

In [ ]:
def numerical_gradient(fun, w, eps=1e-6):
    grad = np.zeros_like(w)
    for k in range(len(w)):
        e_k = np.zeros_like(w)
        e_k[k] = eps
        grad[k] = (fun(w + e_k) - fun(w - e_k)) / (2 * eps)
    return grad


w_test = rng.normal(scale=0.1, size=X_gd.shape[1])
b_test = 0.1

_, grad_analytical, _ = cost_and_gradient(w_test, b_test, X_gd, y_train)
grad_numerical = numerical_gradient(lambda w: cost_and_gradient(w, b_test, X_gd, y_train)[0], w_test)

relative_error = np.linalg.norm(grad_analytical - grad_numerical) / np.linalg.norm(grad_analytical + grad_numerical)
print("analytical:", np.round(grad_analytical, 6))
print("numerical :", np.round(grad_numerical, 6))
print(f"relative error: {relative_error:.2e}")

## 2.4 A Gradient Descent Trainer

One function covers all three variants: `batch_size=None` gives batch GD, `batch_size=1` gives SGD, and anything in between gives mini-batch GD. The trainer records the training cost (and, optionally, the validation log-loss) after every epoch.

In [ ]:
def fit_logistic_gd(X, y, lr, n_epochs, batch_size=None, l2=0.0, X_val=None, y_val=None, seed=RANDOM_STATE):
    gen = np.random.default_rng(seed)
    n, d = X.shape
    w, b = np.zeros(d), 0.0
    batch_size = n if batch_size is None else batch_size
    history = {"train": [], "val": []}

    for _ in range(n_epochs):
        order = gen.permutation(n)  # shuffle before every epoch
        for start in range(0, n, batch_size):
            idx = order[start : start + batch_size]
            _, grad_w, grad_b = cost_and_gradient(w, b, X[idx], y[idx], l2)
            w -= lr * grad_w
            b -= lr * grad_b
        history["train"].append(binary_log_loss(y, sigmoid(X @ w + b)).mean())
        if X_val is not None:
            history["val"].append(binary_log_loss(y_val, sigmoid(X_val @ w + b)).mean())
    return w, b, history


# Reference points: the exact optimum (scikit-learn, no penalty) and a model that always predicts the default rate
reference = LogisticRegression(C=np.inf, max_iter=5000).fit(X_gd, y_train)
optimum_loss = log_loss(y_train, reference.predict_proba(X_gd)[:, 1])
prior_loss = log_loss(y_train, np.full(len(y_train), y_train.mean()))
print(f"log-loss of the optimum (sklearn): {optimum_loss:.4f}")
print(f"log-loss of the constant model   : {prior_loss:.4f}")

## 2.5 Feature Scaling Decides Whether Gradient Descent Works

🔁 **Recap.** In ML I we scaled features for KNN and SVM because of *distances*. For gradient descent there is a second reason: the gradient for $w_k$ is proportional to the feature $x_k$. With `LIMIT_BAL` in the hundreds of thousands and `PAY_0` between −2 and 8, the cost surface is a long, narrow valley. A learning rate small enough not to explode along the steep direction is far too small along the flat one.

We run 200 epochs of batch GD on the **raw** features with three learning rates, and on the **standardised** features with one.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for lr in [1e-10, 1e-9, 1e-8]:
    _, _, hist = fit_logistic_gd(X_gd_raw, y_train, lr=lr, n_epochs=200)
    ax.plot(hist["train"], label=f"raw features, lr = {lr:g}")
_, _, hist = fit_logistic_gd(X_gd, y_train, lr=1.0, n_epochs=200)
ax.plot(hist["train"], label="standardised features, lr = 1", linewidth=2.5, color="black")

ax.axhline(optimum_loss, linestyle="--", color="green", label="optimum (sklearn)")
ax.axhline(prior_loss, linestyle=":", color="grey", label="constant model")
ax.set(
    xlabel="epoch",
    ylabel="training log-loss",
    title="Batch gradient descent: raw vs. standardised features",
    ylim=(0.44, 1.0),
)
ax.legend()
plt.show()

With raw features no learning rate works within the budget: the smallest one never gets below the loss of the constant model, while the larger ones make the loss jump around far above it (the curve for `lr = 1e-08` does not even fit on the chart). After standardisation the same algorithm reaches the optimum. Scaling is part of the optimisation problem, not just a matter of taste.

## 2.6 The Learning Rate

On the standardised features we now compare four learning rates.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for lr in [0.1, 1.0, 5.0, 30.0]:
    _, _, hist = fit_logistic_gd(X_gd, y_train, lr=lr, n_epochs=100)
    ax.plot(hist["train"], label=f"lr = {lr:g}")
ax.axhline(optimum_loss, linestyle="--", color="black", label="optimum (sklearn)")
ax.set(
    xlabel="epoch",
    ylabel="training log-loss",
    title="Batch gradient descent: the effect of the learning rate",
    ylim=(0.44, 1.0),
)
ax.legend()
plt.show()

- **Too small** (0.1): every step goes in the right direction, but convergence is slow and the budget runs out before the optimum.
- **Reasonable** (1 and 5): the loss decreases quickly and settles at the optimum. With `lr = 5` this happens within the first epoch, so its curve lies on top of the dashed line.
- **Too large** (30): the steps overshoot the minimum and the loss jumps up and down instead of decreasing.

🔭 **Why this matters in ML II.** The learning rate will come back several times, as a key hyperparameter of models that are trained step by step. The trade-off you see here (small rate: slow but stable; large rate: fast but unstable) is the intuition to bring to the first lecture.

## 2.7 Batch vs. Mini-Batch vs. Stochastic Gradient Descent

We now train for 10 epochs with different batch sizes. Smaller batches give noisier gradients, so they need a smaller learning rate. The values below were chosen to work reasonably for each batch size.

In [ ]:
configs = [
    ("batch (B = n)", None, 1.0),
    ("mini-batch (B = 512)", 512, 0.1),
    ("mini-batch (B = 32)", 32, 0.05),
    ("stochastic (B = 1)", 1, 0.01),
]

fig, ax = plt.subplots(figsize=(9, 4.5))
rows = []
for name, batch_size, lr in configs:
    start = time.perf_counter()
    _, _, hist = fit_logistic_gd(X_gd, y_train, lr=lr, n_epochs=10, batch_size=batch_size, X_val=X_gd_val, y_val=y_val)
    elapsed = time.perf_counter() - start
    n_updates = 10 * int(np.ceil(len(y_train) / (batch_size or len(y_train))))
    ax.plot(range(1, 11), hist["train"], marker="o", label=f"{name}, lr = {lr:g}")
    rows.append(
        {
            "variant": name,
            "updates": n_updates,
            "train log-loss": hist["train"][-1],
            "val log-loss": hist["val"][-1],
            "time [s]": elapsed,
        }
    )
ax.axhline(optimum_loss, linestyle="--", color="green", label="optimum (sklearn)")
ax.set(xlabel="epoch", ylabel="training log-loss", title="Training cost after each epoch", ylim=(0.46, 0.53))
ax.legend()
plt.show()

pd.DataFrame(rows).set_index("variant")

- **Batch GD** makes one update per epoch. After 10 epochs it is still clearly above the optimum.
- **Mini-batches** make many cheap updates per epoch and are close to the optimum after a few epochs.
- **SGD** makes the most updates, but each one is based on a single client. The loss does not decrease monotonically; it fluctuates around the optimum. In pure Python it is also by far the slowest, because the loop runs once per observation.

🔭 **Why this matters in ML II.** "Epoch", "batch size", "learning rate" and "shuffle" will appear in the very first neural network lab. They mean exactly what you saw here.

## 2.8 The Negative Gradient With Respect to the Prediction

One more derivative is worth remembering. Instead of differentiating with respect to the weights, differentiate the loss with respect to the **prediction itself**:

| Loss | $-\dfrac{\partial L}{\partial \hat y}$ (or $-\dfrac{\partial L}{\partial z}$) | Interpretation |
|---|---|---|
| $\frac{1}{2}(y - \hat y)^2$ | $y - \hat y$ | the ordinary **residual** |
| $\lvert y - \hat y \rvert$ | $\operatorname{sign}(y - \hat y)$ | the sign of the residual |
| log-loss, $p = \sigma(z)$ | $y - p$ | the "residual" on the probability scale |

The negative gradient says in which direction, and how strongly, each individual prediction should move to reduce the loss. We check the first and the third row numerically.

In [ ]:
y_demo = np.array([3.0, -1.0, 0.5])
y_hat_demo = np.array([2.0, 0.5, 0.5])
neg_grad_sq = -numerical_gradient(lambda yh: squared_loss(y_demo, yh).sum(), y_hat_demo)
print(
    "squared loss  - negative gradient:", np.round(neg_grad_sq, 6) + 0.0, "| residual y - y_hat:", y_demo - y_hat_demo
)

y_bin = np.array([1.0, 0.0, 1.0])
z_demo = np.array([0.3, 2.0, -1.5])
neg_grad_log = -numerical_gradient(lambda z: binary_log_loss(y_bin, sigmoid(z)).sum(), z_demo)
print("log-loss      - negative gradient:", np.round(neg_grad_log, 6), "| y - p:", np.round(y_bin - sigmoid(z_demo), 6))

🔭 **Why this matters in ML II.** Keep this table in mind for the first lecture. One of the most important ideas of ML II is built around "the negative gradient of the loss with respect to the current predictions".

**Exercise 2 (vectorised softmax).** Multinomial logistic regression (Lecture 1) uses the softmax $p_{ik} = \dfrac{e^{z_{ik}}}{\sum_j e^{z_{ij}}}$ and the cross-entropy $J = -\frac{1}{n}\sum_i \sum_k y_{ik} \log p_{ik}$, where $Y$ is the one-hot encoded target.

1. Write `softmax(Z)` for a matrix `Z` of shape `(n, K)`, **without loops**. Make it numerically stable by subtracting the row-wise maximum of `Z` before exponentiating. Why does this not change the result?
2. Check that `softmax(np.array([[1000.0, 1001.0, 1002.0]]))` returns finite numbers.
3. Using `Z = rng.normal(size=(6, 3))` and `labels = np.array([0, 2, 1, 1, 0, 2])`, compute the cross-entropy and compare it with `log_loss(labels, softmax(Z))`.
4. Verify numerically that the gradient of $n \cdot J$ with respect to `Z` equals $P - Y$ (hint: flatten `Z` to use `numerical_gradient`).

In [ ]:
# your solution here

<details>
<summary>Solution</summary>

```python
def softmax(Z):
    Z_shifted = Z - Z.max(axis=1, keepdims=True)  # e^(z - c) / sum e^(z - c) = e^z / sum e^z
    expZ = np.exp(Z_shifted)
    return expZ / expZ.sum(axis=1, keepdims=True)  # broadcasting: (n, K) / (n, 1)


print(softmax(np.array([[1000.0, 1001.0, 1002.0]])))  # finite; the naive version returns nan

Z = rng.normal(size=(6, 3))
labels = np.array([0, 2, 1, 1, 0, 2])
Y = np.eye(3)[labels]  # one-hot encoding, shape (6, 3)
P = softmax(Z)
cross_entropy = -np.mean(np.sum(Y * np.log(P), axis=1))
print(cross_entropy, log_loss(labels, P))


def total_ce(z_flat):
    return -np.sum(Y * np.log(softmax(z_flat.reshape(Z.shape))))


grad_num = numerical_gradient(total_ce, Z.ravel()).reshape(Z.shape)
print(np.abs(grad_num - (P - Y)).max())  # ~1e-9: the gradient w.r.t. the scores is P - Y
```

</details>

# 3. Regularization

🔁 **Recap (Lecture 4).** Regularization adds a penalty on the size of the parameters to the cost:

$$J(w) = \frac{1}{n}\sum_{i=1}^n L(y_i, \hat y_i) + \lambda R(w), \qquad R(w) = \tfrac{1}{2}\lVert w \rVert_2^2 \ \text{(L2, Ridge)} \quad \text{or} \quad \lVert w \rVert_1 \ \text{(L1, Lasso)}.$$

For the L2 penalty the gradient gains the term $\lambda w$, so one gradient descent step becomes

$$w \leftarrow w - \eta\big(\nabla L + \lambda w\big) = (1 - \eta\lambda)\, w - \eta \nabla L .$$

Before each step the weights are multiplied by $(1 - \eta\lambda) < 1$, i.e. they **shrink towards zero**. This is why the L2 penalty is also called **weight decay**. In `cost_and_gradient` this is the `l2 * w` term. Note that in scikit-learn the strength of the penalty is $\lambda$ (`alpha`) for linear regression, but $C \propto 1/\lambda$ for logistic regression and SVM.

To see regularization at work we need a model that *can* overfit. We take only **300 clients**, expand seven features into all polynomial terms up to degree 3 (119 features), and train with GD for 1,000 epochs with three penalty strengths. This overfitting setup is reused in Section 4.

In [ ]:
POLY_FEATURES = ["LIMIT_BAL", "AGE", "PAY_0", "PAY_2", "BILL_AMT1", "PAY_AMT1", "PAY_AMT2"]
small_idx = np.random.default_rng(RANDOM_STATE).choice(
    len(X_train), size=300, replace=False
)  # own generator: reproducible

poly = PolynomialFeatures(degree=3, include_bias=False)
scaler_poly = StandardScaler()
X_small = scaler_poly.fit_transform(poly.fit_transform(X_train[POLY_FEATURES].to_numpy(dtype=float)[small_idx]))
y_small = y_train[small_idx]
# extreme validation clients could produce huge polynomial values; we clip them to +-10 standard deviations
X_small_val = np.clip(scaler_poly.transform(poly.transform(X_val[POLY_FEATURES].to_numpy(dtype=float))), -10, 10)
print("training matrix:", X_small.shape, "| validation matrix:", X_small_val.shape)

histories, weights = {}, {}
for l2 in [0.0, 0.01, 0.1]:
    w, _, hist = fit_logistic_gd(X_small, y_small, lr=0.5, n_epochs=1000, l2=l2, X_val=X_small_val, y_val=y_val)
    histories[l2], weights[l2] = hist, w

pd.DataFrame(
    {
        "final train log-loss": {l2: h["train"][-1] for l2, h in histories.items()},
        "final val log-loss": {l2: h["val"][-1] for l2, h in histories.items()},
        "||w||_2": {l2: np.linalg.norm(w) for l2, w in weights.items()},
    }
).rename_axis("lambda (l2)")

Without a penalty the model fits the 300 training clients much better and the validation clients much worse. A larger $\lambda$ gives a higher training loss, a lower validation loss and a smaller weight vector.

🔭 **Why this matters in ML II.** L1 and L2 penalties are not specific to linear models. You will meet the same knobs under names such as `reg_lambda`, `reg_alpha` or `weight_decay`, often in models with no single "coefficient vector". The mechanics are the same: a penalty in the cost, a shrinkage term in the update.

# 4. Bias–Variance Trade-off and Training Curves

🔁 **Recap (Lecture 2).** For squared loss the expected prediction error at a point decomposes as

$$\mathbb{E}\big[(Y - \hat f(x))^2\big] = \underbrace{\big(\mathbb{E}[\hat f(x)] - f(x)\big)^2}_{\text{bias}^2} + \underbrace{\operatorname{Var}\big(\hat f(x)\big)}_{\text{variance}} + \underbrace{\sigma^2}_{\text{noise}} .$$

Simple models have high bias; flexible models have high variance. We diagnose where a model stands with **curves**:

- a **validation curve**: training and validation score as a function of a complexity hyperparameter;
- a **learning curve over time**: training and validation loss as a function of the number of training iterations (epochs);
- a **learning curve over data**: the same as a function of the training-set size (see `core_ml_techniques.ipynb`, Section 9.4).

## 4.1 Validation Curve: the Depth of a Tree

In [ ]:
depths = [1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20]
auc_train, auc_val = [], []
for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE).fit(X_train, y_train)
    auc_train.append(roc_auc_score(y_train, tree.predict_proba(X_train)[:, 1]))
    auc_val.append(roc_auc_score(y_val, tree.predict_proba(X_val)[:, 1]))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(depths, auc_train, marker="o", label="training")
ax.plot(depths, auc_val, marker="o", label="validation")
best_depth = depths[int(np.argmax(auc_val))]
ax.axvline(best_depth, linestyle="--", color="grey", label=f"best validation depth = {best_depth}")
ax.set(xlabel="max_depth", ylabel="ROC AUC", title="Validation curve of a decision tree")
ax.legend()
plt.show()

Shallow trees underfit (both scores low: high bias). Beyond the best depth the training score keeps rising towards 1 while the validation score falls (high variance). The gap between the two curves is the clearest symptom of overfitting.

## 4.2 Training Curves Over Time

The number of gradient descent iterations is also a complexity knob. Here are the training and validation curves of the three models from Section 3.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
for ax, (l2, hist) in zip(axes, histories.items()):
    best_epoch = int(np.argmin(hist["val"]))
    ax.plot(hist["train"], label="training")
    ax.plot(hist["val"], label="validation")
    ax.axvline(best_epoch, linestyle="--", color="grey", label=f"min. validation loss (epoch {best_epoch})")
    ax.set(xlabel="epoch", title=f"lambda = {l2:g}", ylim=(0.25, 0.75))
    ax.legend()
axes[0].set_ylabel("log-loss")
plt.tight_layout()
plt.show()

- Without a penalty, the validation loss reaches its minimum early in training and then **rises** again, while the training loss keeps falling. The model first learns the signal and then memorises the 300 training clients.
- With a stronger penalty the validation curve flattens out instead of rising.
- Two ways to control the same problem are visible: **penalise** the parameters, or **stop** training at the epoch with the lowest validation loss. The second approach is known as *early stopping*. It is only legitimate if the epoch is selected on validation data, never on the test set.

## 4.3 Averaging Reduces Variance, Not Bias

🔁 **Recap (Lecture 5).** Bagging and Random Forest average many trees. If $B$ trees each have variance $\sigma^2$ and pairwise correlation $\rho$, their average has variance $\rho\sigma^2 + \frac{1 - \rho}{B}\sigma^2$. **Averaging attacks variance.** But what does averaging do to bias?

We estimate bias² and variance by simulation. We draw 50 independent training sets of 150 points from a known function $f(x) = \sin(1.5x) + 0.3x$ with noise $\sigma = 0.4$. On each we fit four models: a **stump** (a tree of depth 1), a **deep tree**, a bagging ensemble of 100 stumps, and a bagging ensemble of 100 deep trees. We then compare the 50 fitted curves at each $x$.

In [ ]:
def true_function(x):
    return np.sin(1.5 * x) + 0.3 * x


NOISE_SD = 0.4
x_grid = np.linspace(0, 5, 200)

model_factories = {
    "stump": lambda: DecisionTreeRegressor(max_depth=1),
    "deep tree": lambda: DecisionTreeRegressor(),
    "bagged stumps": lambda: BaggingRegressor(
        DecisionTreeRegressor(max_depth=1), n_estimators=100, random_state=RANDOM_STATE
    ),
    "bagged deep trees": lambda: BaggingRegressor(DecisionTreeRegressor(), n_estimators=100, random_state=RANDOM_STATE),
}

sim_rng = np.random.default_rng(RANDOM_STATE)
simulated_sets = []
for _ in range(50):
    x_sim = sim_rng.uniform(0, 5, 150)
    simulated_sets.append((x_sim, true_function(x_sim) + sim_rng.normal(0, NOISE_SD, 150)))

predictions, rows = {}, []
for name, make_model in model_factories.items():
    preds = np.array([make_model().fit(xs[:, None], ys).predict(x_grid[:, None]) for xs, ys in simulated_sets])
    predictions[name] = preds
    bias2 = np.mean((preds.mean(axis=0) - true_function(x_grid)) ** 2)
    variance = np.mean(preds.var(axis=0))
    rows.append(
        {
            "model": name,
            "bias^2": bias2,
            "variance": variance,
            "expected MSE = bias^2 + var + noise": bias2 + variance + NOISE_SD**2,
        }
    )

pd.DataFrame(rows).set_index("model")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
for ax, (name, preds) in zip(axes, predictions.items()):
    for curve in preds[:20]:
        ax.plot(x_grid, curve, color="tab:blue", alpha=0.15)
    ax.plot(x_grid, preds.mean(axis=0), color="tab:red", linewidth=2, label="average fit")
    ax.plot(x_grid, true_function(x_grid), color="black", linestyle="--", label="true f(x)")
    ax.set(title=name, xlabel="x")
axes[0].set_ylabel("y")
axes[0].legend(loc="upper left")
plt.tight_layout()
plt.show()

- **Deep tree:** almost no bias (the average fit follows $f$), but every training set gives a very different, jagged curve: high variance.
- **Bagged deep trees** (essentially a Random Forest with one feature): the bias stays low and the variance drops considerably. This is why Random Forest works.
- **Stump:** a single split cannot follow the sine wave, so the bias is large; the variance is small.
- **Bagged stumps:** the variance falls further, but the **bias is practically unchanged**. Averaging 100 models that all make the same systematic error still gives that systematic error.

🔭 **An open question for ML II.** Bagging is a cure for high variance. What could cure the **high bias** of an ensemble of simple trees? Keep the question in mind until the first lecture.

# 5. Decision Trees as Building Blocks

In ML I the decision tree was mainly a step towards the Random Forest. In ML II it becomes the building block of most of the models in the first part of the course, often as a **regression tree**, even when the task is classification. This section revises the details that matter most.

## 5.1 A Regression Tree Is a Step Function

In [ ]:
x_1d = sim_rng.uniform(0, 5, 200)
y_1d = true_function(x_1d) + sim_rng.normal(0, NOISE_SD, 200)

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
for ax, depth in zip(axes, [1, 2, 4, 10]):
    tree = DecisionTreeRegressor(max_depth=depth, random_state=RANDOM_STATE).fit(x_1d[:, None], y_1d)
    ax.scatter(x_1d, y_1d, s=8, alpha=0.4, color="grey")
    ax.plot(x_grid, tree.predict(x_grid[:, None]), color="tab:red", linewidth=2)
    ax.set(title=f"max_depth = {depth} ({tree.get_n_leaves()} leaves)", xlabel="x")
axes[0].set_ylabel("y")
plt.tight_layout()
plt.show()

🔁 **Recap.** A tree splits the feature space into regions (leaves) $R_1, \dots, R_M$ and predicts a constant in each: $\hat f(x) = \sum_m c_m \mathbb{1}(x \in R_m)$. A depth-$d$ tree has at most $2^d$ leaves. A depth-1 tree (stump) has two.

## 5.2 Split Search From Scratch

🔁 **Recap.** For squared error, CART chooses the split (feature $j$, threshold $s$) that minimises the total within-leaf sum of squared errors:

$$\min_{j,\, s} \Big[ \sum_{x_i \in R_L(j,s)} (y_i - \bar y_L)^2 + \sum_{x_i \in R_R(j,s)} (y_i - \bar y_R)^2 \Big].$$

The from-scratch implementation in `decision_trees_and_random_forest.ipynb` tried every threshold in a loop. With **cumulative sums** the search over all thresholds of one feature costs a single sort plus $O(n)$ operations. It uses the identity $\sum (y_i - \bar y)^2 = \sum y_i^2 - (\sum y_i)^2 / n$.

In [ ]:
def best_split_1d(x, y):
    # Best squared-error split of one feature; returns (threshold, SSE after the split).
    order = np.argsort(x)
    x_sorted, y_sorted = x[order], y[order]
    n = len(y)

    n_left = np.arange(1, n)  # observations in the left child for each cut position
    sum_left = np.cumsum(y_sorted)[:-1]
    sq_left = np.cumsum(y_sorted**2)[:-1]
    sum_right = y_sorted.sum() - sum_left
    sq_right = (y_sorted**2).sum() - sq_left
    n_right = n - n_left

    sse = (sq_left - sum_left**2 / n_left) + (sq_right - sum_right**2 / n_right)
    sse[x_sorted[1:] == x_sorted[:-1]] = np.inf  # cannot cut between two equal values

    k = int(np.argmin(sse))
    threshold = (x_sorted[k] + x_sorted[k + 1]) / 2  # midpoint, as in scikit-learn
    return threshold, sse[k]


threshold, sse = best_split_1d(x_1d, y_1d)
stump = DecisionTreeRegressor(max_depth=1).fit(x_1d[:, None], y_1d)
print(f"from scratch : threshold = {threshold:.6f}, SSE = {sse:.4f}")
print(f"scikit-learn : threshold = {stump.tree_.threshold[0]:.6f}")

## 5.3 The Value in a Leaf Minimises the Loss in That Leaf

The constant $c_m$ stored in a leaf is not arbitrary. It is the value that **minimises the loss** over the training observations in that leaf:

- squared error: $\arg\min_c \sum (y_i - c)^2 = \text{mean}$;
- absolute error: $\arg\min_c \sum \lvert y_i - c \rvert = \text{median}$.

We check this on a tree trained with `criterion="absolute_error"` on data with fat-tailed (Student-t) noise, where the mean and the median of a leaf differ visibly.

In [ ]:
y_fat = true_function(x_1d) + 0.4 * sim_rng.standard_t(df=2, size=len(x_1d))
tree_mae = DecisionTreeRegressor(max_depth=2, criterion="absolute_error", random_state=RANDOM_STATE).fit(
    x_1d[:, None], y_fat
)

leaf_of = tree_mae.apply(x_1d[:, None])
pd.DataFrame(
    [
        {
            "leaf": leaf,
            "n": int(np.sum(leaf_of == leaf)),
            "value stored by sklearn": tree_mae.tree_.value[leaf].item(),
            "median of y in leaf": np.median(y_fat[leaf_of == leaf]),
            "mean of y in leaf": y_fat[leaf_of == leaf].mean(),
        }
        for leaf in np.unique(leaf_of)
    ]
).set_index("leaf")

🔭 **Why this matters in ML II.** "Which constant should a leaf predict?" has a different answer for every loss function. In ML II you will meet trees whose leaf values come from a loss-specific formula rather than a plain mean.

## 5.4 A Regression Tree on a 0/1 Target Gives Probabilities

For a binary target $y \in \{0, 1\}$, the mean of $y$ in a leaf is the **share of positives** — a probability estimate. The squared-error impurity of a node with share $p$ is $p(1 - p)$, which is exactly half of the Gini impurity $2p(1 - p)$. A `DecisionTreeRegressor` on the 0/1 target therefore chooses **the same splits** as a `DecisionTreeClassifier` with the Gini criterion, and predicts the same probabilities:

In [ ]:
tree_reg = DecisionTreeRegressor(max_depth=4, random_state=RANDOM_STATE).fit(X_train, y_train)
tree_clf = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE).fit(X_train, y_train)

p_reg = tree_reg.predict(X_val)
p_clf = tree_clf.predict_proba(X_val)[:, 1]
print(f"max |difference| between the two sets of probabilities: {np.abs(p_reg - p_clf).max():.2e}")
print(f"ROC AUC - regressor: {roc_auc_score(y_val, p_reg):.4f}, classifier: {roc_auc_score(y_val, p_clf):.4f}")

## 5.5 Observation Weights: `sample_weight`

🔁 **Recap (Lecture 4, class rebalancing).** Every scikit-learn tree accepts `sample_weight` in `fit`. An observation with weight 3 counts as three identical observations: in the impurity of each candidate split, in the leaf values and in `min_samples_leaf`-type constraints (via `min_weight_fraction_leaf`). `class_weight="balanced"` is nothing more than a shortcut that sets the weight of each observation to $n / (K \cdot n_k)$, where $n_k$ is the size of its class:

In [ ]:
weights_balanced = compute_sample_weight("balanced", y_train)
print("weight of a non-defaulter:", weights_balanced[y_train == 0][0].round(4))
print("weight of a defaulter    :", weights_balanced[y_train == 1][0].round(4))

tree_cw = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=RANDOM_STATE).fit(X_train, y_train)
tree_sw = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE).fit(
    X_train, y_train, sample_weight=weights_balanced
)
print(
    "class_weight='balanced' vs. explicit sample_weight - max |difference|:",
    np.abs(tree_cw.predict_proba(X_val) - tree_sw.predict_proba(X_val)).max(),
)

features = X_train.columns
print("\nfirst split without weights:", features[tree_clf.tree_.feature[0]], "<=", tree_clf.tree_.threshold[0])
print("first split with weights   :", features[tree_cw.tree_.feature[0]], "<=", tree_cw.tree_.threshold[0])

The weights change **the structure of the tree** (here, already the first split), not only the predicted probabilities: misclassifying a heavily weighted observation is more expensive, so the tree works harder to isolate it.

🔭 **Why this matters in ML II.** Several ML II models train trees on **re-weighted** data. Make sure the mechanics of `sample_weight` are clear.

**Exercise 3 (best split over all features).** Using `best_split_1d`, write `best_split(X, y)` that returns the best feature name, threshold and SSE over **all** columns of a DataFrame `X`. Apply it to the credit-card training data with the 0/1 target `y_train` (squared error on a 0/1 target = Gini, see 5.4). Compare your result with the first split of `tree_clf`.

In [ ]:
# your solution here

<details>
<summary>Solution</summary>

```python
def best_split(X, y):
    best = (None, None, np.inf)
    for col in X.columns:
        threshold, sse = best_split_1d(X[col].to_numpy(dtype=float), y.astype(float))
        if sse < best[2]:
            best = (col, threshold, sse)
    return best


print(best_split(X_train, y_train))
print(features[tree_clf.tree_.feature[0]], tree_clf.tree_.threshold[0])  # same feature and threshold
```

</details>

# 6. A Leakage-Free Benchmark Pipeline

We now put the pieces together on the full credit-card dataset and build **ML I benchmarks** for ML II. The checklist is the one from the Kaggle League:

1. Stateless cleaning and feature engineering (row-wise, no information from other rows) may be done before the split.
2. Everything **stateful** (scalers, encoders, imputers, resamplers) lives **inside a `Pipeline`**, so it is fitted only on the training folds.
3. Hyperparameters are tuned with **cross-validation on the training set**. The validation set compares the models, and the test set is used **once**, at the end.
4. Always report a **naive baseline**.

## 6.1 Stateless Cleaning and Feature Engineering

As discussed in `eda_and_econometric_models.ipynb` (homework), `EDUCATION` and `MARRIAGE` contain undocumented codes. We merge them into the "other" category. We also add three simple, economically motivated features.

In [ ]:
def prepare_features(frame):
    frame = frame.copy()
    frame["EDUCATION"] = frame["EDUCATION"].replace({0: 4, 5: 4, 6: 4})  # undocumented codes -> "others"
    frame["MARRIAGE"] = frame["MARRIAGE"].replace({0: 3})
    frame["UTILISATION_1"] = frame["BILL_AMT1"] / frame["LIMIT_BAL"]  # share of the limit in use
    frame["PAY_RATIO_1"] = frame["PAY_AMT1"] / frame["BILL_AMT1"].clip(lower=1)  # payment relative to the bill
    pay_status = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
    frame["N_MONTHS_LATE"] = (frame[pay_status] > 0).sum(axis=1)  # months with a payment delay
    return frame


Xf_train, Xf_val, Xf_test = (prepare_features(part) for part in (X_train, X_val, X_test))

CATEGORICAL = ["SEX", "EDUCATION", "MARRIAGE"]
NUMERICAL = [col for col in Xf_train.columns if col not in CATEGORICAL]
print(f"{len(NUMERICAL)} numerical and {len(CATEGORICAL)} categorical features")

## 6.2 Two Preprocessors for Two Model Families

🔁 **Recap.** Gradient-based and distance-based models (logistic regression, SVM, KNN) need **scaled** numerical features. Trees are invariant to monotone transformations of a feature, so scaling changes nothing for them. Both need categorical features encoded as numbers.

In [ ]:
linear_preprocessor = ColumnTransformer(
    [
        ("num", StandardScaler(), NUMERICAL),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    ]
)
tree_preprocessor = ColumnTransformer(
    [
        ("num", "passthrough", NUMERICAL),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
    ]
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 6.3 Tuning With Cross-Validation on the Training Set

The penalty $C$ is searched on a **log scale**. For the Random Forest we tune the two hyperparameters that matter most (`min_samples_leaf`, `max_features`) and keep `n_estimators` fixed. To keep the class fast we use 200 trees, each grown on a 50% bootstrap sample (`max_samples=0.5`), and 3-fold CV. The Random Forest search takes about 20–30 seconds.

In [ ]:
logit_search = RandomizedSearchCV(
    Pipeline([("pre", linear_preprocessor), ("model", LogisticRegression(max_iter=2000))]),
    param_distributions={"model__C": loguniform(1e-3, 1e2)},
    n_iter=10,
    cv=cv,
    scoring="roc_auc",
    random_state=RANDOM_STATE,
    n_jobs=-1,
).fit(Xf_train, y_train)
print(
    f"logistic regression - best C = {logit_search.best_params_['model__C']:.3g}, CV ROC AUC = {logit_search.best_score_:.4f}"
)

start = time.perf_counter()
forest_search = RandomizedSearchCV(
    Pipeline(
        [
            ("pre", tree_preprocessor),
            ("model", RandomForestClassifier(n_estimators=200, max_samples=0.5, random_state=RANDOM_STATE, n_jobs=-1)),
        ]
    ),
    param_distributions={"model__min_samples_leaf": randint(1, 100), "model__max_features": uniform(0.1, 0.6)},
    n_iter=10,
    cv=3,
    scoring="roc_auc",
    random_state=RANDOM_STATE,
).fit(Xf_train, y_train)
best_rf = forest_search.best_params_
print(
    f"random forest       - best min_samples_leaf = {best_rf['model__min_samples_leaf']}, "
    f"max_features = {best_rf['model__max_features']:.2f}, "
    f"CV ROC AUC = {forest_search.best_score_:.4f} ({time.perf_counter() - start:.0f} s)"
)

## 6.4 Comparing Models on the Validation Set

🔁 **Recap (Lecture 2).** We report four probability metrics, each answering a different question:

| Metric | Question | Better |
|---|---|---|
| ROC AUC | Does the model **rank** defaulters above non-defaulters? | higher; 0.5 = random |
| PR AUC (average precision) | How precise is the model among the clients it flags, across thresholds? Informative under class imbalance. | higher; baseline = default rate |
| Log-loss | How good are the **probabilities**, punishing confident mistakes? | lower |
| Brier score | Mean squared error of the probabilities | lower |

In [ ]:
def probability_metrics(y_true, p):
    return {
        "ROC AUC": roc_auc_score(y_true, p),
        "PR AUC": average_precision_score(y_true, p),
        "log-loss": log_loss(y_true, p),
        "Brier": brier_score_loss(y_true, p),
    }


candidates = {
    "naive baseline (default rate)": DummyClassifier(strategy="prior").fit(Xf_train, y_train),
    "logistic regression (tuned)": logit_search.best_estimator_,
    "random forest (tuned)": forest_search.best_estimator_,
}
val_results = pd.DataFrame(
    {name: probability_metrics(y_val, model.predict_proba(Xf_val)[:, 1]) for name, model in candidates.items()}
).T
val_results

A model that ranks well does not necessarily produce good probabilities, so we also look at the **calibration curves** (Lecture 4): among clients with a predicted probability of about 30%, roughly 30% should default.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6))
for name in ["logistic regression (tuned)", "random forest (tuned)"]:
    CalibrationDisplay.from_estimator(candidates[name], Xf_val, y_val, n_bins=10, strategy="quantile", name=name, ax=ax)
ax.set_title("Calibration on the validation set")
plt.show()

## 6.5 The Final Check on the Test Set — and the ML I Benchmarks

Only now, with the model **chosen on the validation set**, do we look at the test set, once. To make it a fair reference for ML II, the selected model is refitted on the training **and** validation data (with the tuned hyperparameters) before scoring.

In [ ]:
best_name = val_results["ROC AUC"].idxmax()
print("selected on the validation set:", best_name)

Xf_train_val = pd.concat([Xf_train, Xf_val])
y_train_val = np.concatenate([y_train, y_val])

ml1_benchmarks = {}
for name, model in candidates.items():
    refitted = clone(model).fit(Xf_train_val, y_train_val)  # clone keeps the tuned hyperparameters, drops the fit
    ml1_benchmarks[name] = probability_metrics(y_test, refitted.predict_proba(Xf_test)[:, 1])

ml1_benchmarks = pd.DataFrame(ml1_benchmarks).T
ml1_benchmarks

**This table is the reference point for ML II.** With the same seed, the same split and the same features, the boosting models and neural networks of the coming weeks should beat these numbers on the test set.

Note, however, that the test set has now been looked at. In ML II, model selection on this split must still happen on training and validation data only. Otherwise the test result becomes optimistic, for the same reason that tuning on the public leaderboard of the Kaggle League was risky.

**Exercise 4 (cost-based threshold).** The ROC AUC does not depend on the classification threshold, but a credit decision does. Assume that granting a loan to a client who then defaults costs **5**, and rejecting a client who would have repaid costs **1** (the cost matrix of the German Credit data from ML I).

1. Using the **validation** probabilities of the tuned Random Forest (`forest_search.best_estimator_`, fitted on the training set), compute the total cost for thresholds `np.linspace(0.05, 0.95, 91)`.
2. Which threshold minimises the cost? Compare it with the theoretical optimum for calibrated probabilities, $t^* = \frac{c_{FP}}{c_{FP} + c_{FN}} = \frac{1}{1 + 5}$.

In [ ]:
# your solution here

<details>
<summary>Solution</summary>

```python
p_val = forest_search.best_estimator_.predict_proba(Xf_val)[:, 1]

thresholds = np.linspace(0.05, 0.95, 91)
costs = []
for t in thresholds:
    reject = p_val >= t
    cost = 5 * np.sum(~reject & (y_val == 1)) + 1 * np.sum(reject & (y_val == 0))
    costs.append(cost)
best_t = thresholds[int(np.argmin(costs))]
print(f"empirically best threshold: {best_t:.2f}; theory for calibrated probabilities: {1 / 6:.3f}")

plt.plot(thresholds, costs)
plt.axvline(1 / 6, linestyle="--", color="grey")
plt.xlabel("threshold")
plt.ylabel("total cost on the validation set")
plt.show()
```

</details>

# 7. Validation for Time-Ordered Data (Self-Study)

🔁 **Recap (Lecture 2, `core_ml_techniques.ipynb` Section 10.5).** With time-ordered data, a shuffled K-fold lets the model train on the future and test on the past. Worse, neighbouring observations are often almost identical, so the model can "look up" the answer of a validation day from its training neighbours. `TimeSeriesSplit` always trains on the past and validates on the future, and its `gap` argument leaves a buffer between the two.

ML II includes sequence models and time-series forecasting, so we finish with a demonstration of how much this matters. We simulate **pure noise** daily returns: by construction, nothing is predictable. The features are rolling momentum and volatility over the past 20 and 60 days. The target is the return over the **next 20 days**, which is a typical setting in finance.

In [ ]:
n_days, horizon = 2500, 20
returns = pd.Series(sim_rng.normal(0, 0.01, n_days))

ts_features = pd.DataFrame(
    {
        "momentum_20": returns.rolling(20).sum(),
        "momentum_60": returns.rolling(60).sum(),
        "volatility_20": returns.rolling(20).std(),
        "volatility_60": returns.rolling(60).std(),
    }
)
future_return = returns[::-1].rolling(horizon).sum()[::-1].shift(-1)  # sum of returns on days t+1, ..., t+20
ts_data = pd.concat([ts_features, future_return.rename("target")], axis=1).dropna()
X_ts, y_ts = ts_data.drop(columns="target"), ts_data["target"]

ts_model = RandomForestRegressor(n_estimators=200, min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=-1)
for name, splitter in [
    ("shuffled KFold", KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)),
    (f"TimeSeriesSplit (gap = {horizon})", TimeSeriesSplit(n_splits=5, gap=horizon)),
]:
    scores = cross_val_score(ts_model, X_ts, y_ts, cv=splitter, scoring="r2")
    print(f"{name:28s} R^2 per fold: {np.round(scores, 3)}  mean: {scores.mean():.3f}")

The returns are pure noise, so the honest out-of-sample $R^2$ cannot be meaningfully above zero. Yet the shuffled K-fold reports a clearly positive $R^2$. Two consecutive days share 19 of their 20 future returns and have almost the same rolling features, so the forest simply recognises the "neighbours" of each validation day in the training data. `TimeSeriesSplit` with a gap removes this leak and reveals the truth: the $R^2$ is negative.

🔭 **Why this matters in ML II.** Overlapping targets, rolling features and shuffled CV are the most common reason why "great" financial models fail out of sample. The same rule holds for every model you will train in ML II.

# 8. Summary

## Checklist: Before the First ML II Lecture You Should Be Able To...

- [ ] distinguish **loss**, **cost** and **evaluation metric**, and explain why ROC AUC is not used as a training loss;
- [ ] write the gradient of the log-loss with respect to the weights ($\frac{1}{n}X^\top(p - y)$) and with respect to the score ($p - y$), and derive it with the chain rule;
- [ ] check a hand-derived gradient numerically;
- [ ] explain what the **learning rate**, **batch size** and **epoch** are, and recognise a learning rate that is too small or too large from a loss curve;
- [ ] explain why gradient descent needs **scaled** features while trees do not;
- [ ] write the L2-penalised gradient descent update and explain the name *weight decay*;
- [ ] read a validation curve and a training curve over epochs, and point to over- and underfitting;
- [ ] explain why averaging (bagging, Random Forest) reduces **variance** but not **bias**;
- [ ] describe how a regression tree chooses a split, what value it stores in a leaf, and what `sample_weight` does;
- [ ] build a leakage-free `Pipeline`, tune it with CV, select on validation data, and report a baseline and probability metrics;
- [ ] choose a cross-validation scheme for time-ordered data.

## Self-Check Quiz

Ten single-choice questions in the style of the pop quizzes. Answers are hidden below.

1. Gradient descent on unscaled features with very different ranges typically: (a) converges faster; (b) needs a learning rate so small that the flat directions barely move; (c) is unaffected, since the optimum is the same; (d) always diverges.
2. For logistic regression, $\partial L / \partial z$ for one observation equals: (a) $y - p$; (b) $p(1 - p)$; (c) $p - y$; (d) $(p - y)\,x$.
3. Which statement about SGD with batch size 1 is true? (a) The training loss decreases monotonically. (b) It makes one update per epoch. (c) It needs no learning rate. (d) Its updates are noisy, so the loss fluctuates around the minimum.
4. The L2 penalty $\frac{\lambda}{2}\lVert w\rVert^2$ changes the GD update to: (a) $w \leftarrow (1 - \eta\lambda)w - \eta\nabla L$; (b) $w \leftarrow w - \eta\lambda\,\operatorname{sign}(w)$; (c) $w \leftarrow (1 + \eta\lambda)w - \eta\nabla L$; (d) $w \leftarrow w - \eta\nabla L / \lambda$.
5. Bagging 100 decision stumps compared with a single stump mainly: (a) removes the bias; (b) reduces the variance and leaves the bias almost unchanged; (c) increases the variance; (d) increases the bias.
6. A regression tree trained with `criterion="absolute_error"` stores in each leaf: (a) the mean; (b) the median; (c) the mode; (d) the midrange of $y$.
7. A `DecisionTreeRegressor` (squared error) fitted to a 0/1 target: (a) cannot be used for classification; (b) predicts only 0 or 1; (c) predicts the share of positives in each leaf and splits like a Gini classification tree; (d) needs the target to be standardised.
8. Training loss keeps falling while validation loss started rising at epoch 40. The model: (a) underfits after epoch 40; (b) overfits after epoch 40; (c) has too small a learning rate; (d) has converged.
9. Where should `StandardScaler` be fitted in a cross-validated workflow? (a) On the whole dataset before splitting; (b) on the test set; (c) inside the Pipeline, i.e. on the training folds only; (d) on the validation fold.
10. For a model predicting 20-day-ahead returns from rolling features, the most appropriate CV scheme is: (a) shuffled K-fold; (b) stratified K-fold; (c) leave-one-out; (d) `TimeSeriesSplit` with a gap of at least 20 days.

<details>
<summary>Answers</summary>

1 – b, 2 – c, 3 – d, 4 – a, 5 – b, 6 – b, 7 – c, 8 – b, 9 – c, 10 – d.

</details>

## Further Reading

- Deisenroth, M. P., Faisal, A. A., Ong, C. S. (2020). *Mathematics for Machine Learning*. Cambridge University Press — Chapter 5 (vector calculus, the chain rule) and Chapter 7 (continuous optimisation, gradient descent). Free at <https://mml-book.github.io/>.
- Ruder, S. (2016). *An overview of gradient descent optimization algorithms*. <https://arxiv.org/abs/1609.04747> — Sections 1–3 (the GD variants and their challenges).
- James, G., Witten, D., Hastie, T., Tibshirani, R. (2021). *An Introduction to Statistical Learning*, 2nd ed. — Chapter 2.2 (bias–variance), Chapter 5 (resampling), Chapter 8.1–8.2.2 (trees, bagging, Random Forest).

# 9. Warm-Up Assignment (Optional, Before Lecture 1 of ML II)

1. **Linear regression by mini-batch GD.** Derive the gradient of the mean squared error $\frac{1}{2n}\lVert y - Xw - b\rVert^2$, adapt `fit_logistic_gd` to linear regression, and fit it on the standardised numerical features of `data/credit_card_default.csv.gz` with `BILL_AMT1` as the target (exclude the other `BILL_AMT*` columns). Verify your gradient with `numerical_gradient` and compare the coefficients with `sklearn.linear_model.LinearRegression`.
2. **Learning-rate schedule.** Modify the trainer so that the learning rate decays as $\eta_t = \eta_0 / (1 + t / \tau)$ after every epoch. Compare SGD with a constant and a decaying learning rate on the data of Section 2.7. What happens to the fluctuations around the minimum?
3. **Bias and variance of a tree, depth by depth.** Repeat the simulation of Section 4.3 for `DecisionTreeRegressor(max_depth=d)` with $d = 1, \dots, 10$, and plot bias² and variance against $d$. At which depth is their sum the smallest?